# Phase 3: Cue Pattern Development (Regex + TextBlob)

**Goal:** Develop and test each of the 10 working cues individually before full labeling.

**Cues (10):**
1) Suspicious_Sender
2) Generic_Greeting
3) Spelling_Grammar (strict)
4) Urgency
5) Threats
6) Emotional_Appeal (Regex + TextBlob)
7) Too_Good_True
8) Personal_Info_Request
9) Suspicious_Link
10) V_Triad_Score (composite)

**Dropped:** No_Branding, Overall_Design, No_Sender_Details

In [38]:
import pandas as pd
import re
from typing import List, Dict
from urllib.parse import urlparse
import ast

## 1. Load Master Dataset

In [52]:
from pathlib import Path

root = Path.cwd()
master_path = root / 'data' / 'processed' / 'master_emails.csv'
if not master_path.exists():
    master_path = root.parent / 'data' / 'processed' / 'master_emails.csv'

print(f"Using master dataset at: {master_path}")
master_df = pd.read_csv(master_path)
print(f"Loaded master_df: {master_df.shape}")
print(master_df.head(2))

Using master dataset at: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\data\processed\master_emails.csv
Loaded master_df: (250, 7)
   email_id                              subject  \
0         1  Re: New gkrellm 2.0.0, gtk2 version   
1         2                Re: The case for spam   

                                     sender  \
0  Brian Fahrlander <kilroy@kamakiriad.com>   
1            Lucas Gonze <lgonze@panix.com>   

                                                body  \
0  On Mon, 26 Aug 2002 19:14:54 +0200, Matthias S...   
1  Dan Brickley wrote:\n> Except that thanks to t...   

                                      extracted_urls            source  \
0  ['http://ftp.freshrpms.net/pub/freshrpms/testi...  spamassassin_ham   
1          ['http://xent.com/mailman/listinfo/fork']  spamassassin_ham   

   actual_class  
0             0  
1             0  


## 2. Define Regex Patterns

In [40]:
# Suspicious Sender (strict - spoofing patterns only)
SUSPICIOUS_SENDER_PATTERN = re.compile(
    r'(paypa1|amazn|micr0soft|goog1e|faceb00k|appleid|secure-?verify|account-?verify|login-?verify|support-?secure)',
    re.IGNORECASE
)

# Generic Greeting
GENERIC_GREETING_PATTERN = re.compile(
    r'\b(dear (customer|user|member|client|valued customer|account holder)|hello (there|user)|hi there)\b',
    re.IGNORECASE
)

# Spelling/Grammar (FIXED: removed legitimate words, only actual misspellings)
MISSPELLINGS_PATTERN = re.compile(
    r'\b(verifiy|accout|passwrod|securty|imediately|recieve|looser|requird|'
    r'upgarde|subscrption|suspened|comfirm|identitiy)\b',
    re.IGNORECASE
)
HOMOGLYPH_PATTERN = re.compile(r'[0O]{3,}|[1Il]{3,}|[5S]{3,}')  # Require 3+ for stricter detection

# Urgency (context required)
URGENCY_PATTERN = re.compile(
    r'\b(urgent|immediately|asap|immediate action).*(action|verify|confirm|click|update)|'
    r'\b(act now|expires|deadline|time\s*sensitive|don\'t (wait|delay)|final notice|within 24 hours|last chance)\b',
    re.IGNORECASE
)

# Threats
THREATS_PATTERN = re.compile(
    r'\b(account (suspended|locked|disabled|frozen)|access (blocked|limited|restricted)|'
    r'will be (terminated|closed|deleted|removed)|service interruption|legal action|penalt(y|ies)|'
    r'payment failed|overdue|disconnection)\b',
    re.IGNORECASE
)

# Emotional Appeal (regex component)
EMOTIONAL_APPEAL_PATTERN = re.compile(
    r'\b(congratulations|help needed|please help|emergency|dear friend|we need your help|'
    r'exclusive|approved|eligible|special offer|act of kindness|donation)\b',
    re.IGNORECASE
)

# Too Good to Be True
TOO_GOOD_TRUE_PATTERN = re.compile(
    r'\b(win|winner|lottery|prize|free money|gift card|bonus|cashback|reward|pre-?approved|'
    r'100% free|refund|points|upgrade|0% apr)\b',
    re.IGNORECASE
)

# Personal Info Request
PERSONAL_INFO_PATTERN = re.compile(
    r'\b(password|passcode|ssn|social security|credit card|bank account|pin|login|verify identity|'
    r'confirm your identity|billing info|card details|account number|routing number)\b',
    re.IGNORECASE
)

# Suspicious Link indicators
SHORTENER_PATTERN = re.compile(r'\b(bit\.ly|t\.co|tinyurl\.com|goo\.gl|ow\.ly)\b', re.IGNORECASE)
SUSPICIOUS_TLD_PATTERN = re.compile(r'\.(tk|ml|ga|cf|gq|xyz|top)(/|$)', re.IGNORECASE)
BRAND_KEYWORDS = [
    'paypal', 'amazon', 'microsoft', 'google', 'apple', 'netflix', 'chase', 'wellsfargo',
    'bankofamerica', 'facebook', 'instagram', 'linkedin', 'adobe', 'dropbox', 'spotify',
    'walmart', 'comcast', 'americanexpress', 'amex', 'nytimes', 'costco', 'geico'
]
ALLOWED_BRAND_DOMAINS = {
    'paypal.com', 'amazon.com', 'microsoft.com', 'google.com', 'apple.com', 'netflix.com',
    'chase.com', 'wellsfargo.com', 'bankofamerica.com', 'facebook.com', 'instagram.com',
    'linkedin.com', 'adobe.com', 'dropbox.com', 'spotify.com', 'walmart.com', 'comcast.com',
    'americanexpress.com', 'nytimes.com', 'costco.com', 'geico.com'
}
SUSPICIOUS_PATH_PATTERN = re.compile(r'/(verify|login|secure|account|confirm|update)', re.IGNORECASE)

## 3. Cue Functions

In [41]:
def check_suspicious_sender(sender: str) -> int:
    if not sender:
        return 0
    sender_lower = sender.lower()
    if SUSPICIOUS_SENDER_PATTERN.search(sender_lower):
        return 1

    # Brand keyword with non-allowed domain
    for brand in BRAND_KEYWORDS:
        if brand in sender_lower:
            if '@' in sender_lower:
                domain = sender_lower.split('@')[-1].strip()
                if domain and domain not in ALLOWED_BRAND_DOMAINS:
                    return 1
    return 0


def check_spelling_grammar(text: str) -> int:
    # Misspellings: 2+ triggers (strict); homoglyphs require 3+ instances
    misspellings = MISSPELLINGS_PATTERN.findall(text)
    homoglyphs = HOMOGLYPH_PATTERN.findall(text)
    if len(misspellings) >= 2:
        return 1
    return 1 if len(homoglyphs) >= 3 else 0


def extract_urls(text: str) -> List[str]:
    if not isinstance(text, str):
        return []
    url_pattern = re.compile(r'https?://[^\s<>"\']+', re.IGNORECASE)
    return url_pattern.findall(text)


def check_suspicious_links(links: List[str]) -> int:
    if not links:
        return 0
    for link in links:
        # Check for shorteners or suspicious TLDs
        if SHORTENER_PATTERN.search(link) or SUSPICIOUS_TLD_PATTERN.search(link):
            return 1
        try:
            host = urlparse(link).netloc.lower().rstrip('.')  # Remove trailing dots
            path = urlparse(link).path.lower()
            
            # Check if host is in allowed domains (legitimate)
            if host in ALLOWED_BRAND_DOMAINS:
                continue
            
            # Brand keyword in host but not an allowed domain (likely phishing)
            for brand in BRAND_KEYWORDS:
                if brand in host:
                    return 1
            
            # Suspicious path on non-allowed domain
            if SUSPICIOUS_PATH_PATTERN.search(path):
                return 1
            
            # Hyphenated host with security/account keywords
            if any(x in host for x in ['verify', 'secure', 'account', 'login']) and '-' in host:
                return 1
                
        except Exception:
            continue
    return 0

## 4. Emotional Appeal (Regex + TextBlob)

In [42]:
try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except ImportError:
    TEXTBLOB_AVAILABLE = False

def check_emotional_appeal(text: str) -> int:
    if EMOTIONAL_APPEAL_PATTERN.search(text):
        return 1
    if not TEXTBLOB_AVAILABLE:
        return 0
    blob = TextBlob(text[:1000])
    if blob.sentiment.polarity > 0.4 and blob.sentiment.subjectivity > 0.5:
        return 1
    return 0

## 5. Build Curated Sample Set (20 emails)

In [55]:
# 15 benign (ham), 15 phishbowl (real), 15 hybrid (synthetic), 15 plain LLM (synthetic)
sample_df = pd.concat([
    master_df[master_df['source'] == 'spamassassin_ham'].sample(15, random_state=42),
    master_df[master_df['source'] == 'phishbowl'].sample(15, random_state=42),
    master_df[master_df['source'] == 'hybrid_vtriad'].sample(15, random_state=42),
    master_df[master_df['source'] == 'plain_llm'].sample(15, random_state=42),
], ignore_index=True)

sample_df['expected_class'] = sample_df['actual_class']
sample_df['full_text'] = (sample_df['subject'].fillna('') + ' ' + sample_df['body'].fillna('')).str.lower()
sample_df.head()

,email_id,subject,sender,body,extracted_urls,source,actual_class,expected_class,full_text
0,84,"Re: ARRRGHHH Had GPG working, now it doesnt.",Brent Welch <welch@panasas.com>,"If you haven't already, you should enable the ...",['https://listman.redhat.com/mailman/listinfo/...,spamassassin_ham,0,0,"re: arrrghhh had gpg working, now it doesnt. i..."
1,54,Digital service launches with 30 free channels,guardian <rssfeeds@example.com>,*Media:* BBC and Sky-backed service will be la...,[],spamassassin_ham,0,0,digital service launches with 30 free channels...
2,71,Trip Notes,aaronsw <rssfeeds@example.com>,"=== Scissors === \n\nI, like most other people...",[],spamassassin_ham,0,0,"trip notes === scissors === \n\ni, like most o..."
3,46,"""I meditated in a cave for 12 years and now I'...",fark <rssfeeds@example.com>,[IMG: http://www.newsisfree.com/Images/fark/sa...,['http://www.newsisfree.com/Images/fark/salon....,spamassassin_ham,0,0,"""i meditated in a cave for 12 years and now i'..."
4,45,[use Perl] Stories for 2002-09-06,pudge@perl.org,use Perl Daily Newsletter\n\nIn this issue:\n ...,['http://use.perl.org/article.pl?sid=02/09/05/...,spamassassin_ham,0,0,[use perl] stories for 2002-09-06 use perl dai...


## 6. Test Each Cue (Single-Cue Validation)

In [44]:
def test_cue(pattern_or_fn, cue_name: str, df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, row in df.iterrows():
        text = row['full_text']
        sender = str(row['sender'])
        links = row['extracted_urls']
        if isinstance(links, str):
            try:
                links = ast.literal_eval(links)
            except Exception:
                links = []

        if hasattr(pattern_or_fn, 'search'):
            triggered = bool(pattern_or_fn.search(text))
        else:
            triggered = bool(pattern_or_fn(text, sender, links))

        results.append({
            'source': row['source'],
            'expected_class': row['expected_class'],
            'triggered': triggered,
            'correct': (triggered and row['expected_class'] == 1) or (not triggered and row['expected_class'] == 0),
            'preview': row['body'][:80]
        })

    df_res = pd.DataFrame(results)
    benign = df_res[df_res['expected_class'] == 0]
    phishing = df_res[df_res['expected_class'] == 1]

    print(f"\n{'='*60}")
    print(f"CUE: {cue_name}")
    print(f"{'='*60}")
    print(f"Benign False Positives: {benign['triggered'].sum()}/{len(benign)}")
    print(f"Phishing True Positives: {phishing['triggered'].sum()}/{len(phishing)}")
    print(f"Accuracy: {df_res['correct'].mean():.0%}")

    return df_res

In [56]:
# Adapter functions
def cue_suspicious_sender(text, sender, links):
    return check_suspicious_sender(sender)

def cue_spelling_grammar(text, sender, links):
    return check_spelling_grammar(text)

def cue_emotional_appeal(text, sender, links):
    return check_emotional_appeal(text)

def cue_suspicious_links(text, sender, links):
    return check_suspicious_links(links)

# Run tests
test_cue(cue_suspicious_sender, 'Suspicious Sender', sample_df)
test_cue(GENERIC_GREETING_PATTERN, 'Generic Greeting', sample_df)
test_cue(cue_spelling_grammar, 'Spelling/Grammar', sample_df)
test_cue(URGENCY_PATTERN, 'Urgency', sample_df)
test_cue(THREATS_PATTERN, 'Threats', sample_df)
test_cue(cue_emotional_appeal, 'Emotional Appeal', sample_df)
test_cue(TOO_GOOD_TRUE_PATTERN, 'Too Good To Be True', sample_df)
test_cue(PERSONAL_INFO_PATTERN, 'Personal Info Request', sample_df)
test_cue(cue_suspicious_links, 'Suspicious Link', sample_df)


CUE: Suspicious Sender
Benign False Positives: 0/15
Phishing True Positives: 0/45
Accuracy: 25%

CUE: Generic Greeting
Benign False Positives: 0/15
Phishing True Positives: 1/45
Accuracy: 27%

CUE: Spelling/Grammar
Benign False Positives: 0/15
Phishing True Positives: 0/45
Accuracy: 25%

CUE: Urgency
Benign False Positives: 0/15
Phishing True Positives: 2/45
Accuracy: 28%

CUE: Threats
Benign False Positives: 0/15
Phishing True Positives: 0/45
Accuracy: 25%

CUE: Emotional Appeal
Benign False Positives: 0/15
Phishing True Positives: 1/45
Accuracy: 27%

CUE: Too Good To Be True
Benign False Positives: 1/15
Phishing True Positives: 0/45
Accuracy: 23%

CUE: Personal Info Request
Benign False Positives: 1/15
Phishing True Positives: 20/45
Accuracy: 57%

CUE: Suspicious Link
Benign False Positives: 0/15
Phishing True Positives: 28/45
Accuracy: 72%


,source,expected_class,triggered,correct,preview
0,spamassassin_ham,0,False,True,"If you haven't already, you should enable the ..."
1,spamassassin_ham,0,False,True,*Media:* BBC and Sky-backed service will be la...
2,spamassassin_ham,0,False,True,"=== Scissors === \n\nI, like most other people..."
3,spamassassin_ham,0,False,True,[IMG: http://www.newsisfree.com/Images/fark/sa...
4,spamassassin_ham,0,False,True,use Perl Daily Newsletter\n\nIn this issue:\n ...
5,spamassassin_ham,0,False,True,"Guys, the Habeas Infringers List (HIL) exists ..."
6,spamassassin_ham,0,False,True,"Hi,\n\nI was inspired by a mode of operation s..."
7,spamassassin_ham,0,False,True,"Matthias Saou wrote:\n\n>Once upon a time, Roi..."
8,spamassassin_ham,0,False,True,"Hi\n\nOn Wed, 4 Sep 2002, Yannick Gingras wrot..."
9,spamassassin_ham,0,False,True,"On Mon, 26 Aug 2002 19:14:54 +0200, Matthias S..."


## 7. V_Triad_Score (Composite - Content + Link Cues)

In [31]:
def compute_v_triad_score(row: pd.Series) -> int:
    text = row['full_text']
    links = row['extracted_urls']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except Exception:
            links = []

    urgency = 1 if URGENCY_PATTERN.search(text) else 0
    threats = 1 if THREATS_PATTERN.search(text) else 0
    too_good = 1 if TOO_GOOD_TRUE_PATTERN.search(text) else 0
    personal = 1 if PERSONAL_INFO_PATTERN.search(text) else 0
    emotional = check_emotional_appeal(text)
    link = check_suspicious_links(links)

    return urgency + threats + too_good + personal + emotional + link

sample_df['v_triad_score'] = sample_df.apply(compute_v_triad_score, axis=1)
print(sample_df[['source', 'expected_class', 'v_triad_score']].head(10))

             source  expected_class  v_triad_score
0  spamassassin_ham               0              0
1  spamassassin_ham               0              0
2  spamassassin_ham               0              0
3  spamassassin_ham               0              0
4  spamassassin_ham               0              0
5         phishbowl               1              0
6         phishbowl               1              0
7         phishbowl               1              1
8         phishbowl               1              2
9         phishbowl               1              2


## 8. Checklist Reminder

- [ ] Tested on 5 benign samples → triggers 0-1 times
- [ ] Tested on 5 phishing samples → triggers 2-3 times
- [ ] Uses re.IGNORECASE where case doesn’t matter
- [ ] Escapes special characters in literal patterns
- [ ] Broad words include context (action verbs)
- [ ] Character-level checks require 2+ instances
- [ ] Inline comments explaining each pattern

## 9. Debug: Inspect False Positives (Ham emails triggering Spelling/Grammar)

In [32]:
# Inspect ham emails that triggered spelling/grammar
ham_sample = sample_df[sample_df['expected_class'] == 0].copy()
ham_sample['spelling_triggered'] = ham_sample.apply(
    lambda row: check_spelling_grammar(row['full_text']),
    axis=1
)

fp_ham = ham_sample[ham_sample['spelling_triggered'] == 1]
print(f"Ham emails triggering Spelling/Grammar: {len(fp_ham)}/{len(ham_sample)}\n")

for idx, row in fp_ham.iterrows():
    print(f"{'='*80}")
    print(f"Source: {row['source']}")
    print(f"Subject: {row['subject']}")
    print(f"Body preview: {row['body'][:300]}")
    
    # Find what triggered
    misspellings = MISSPELLINGS_PATTERN.findall(row['full_text'])
    homoglyphs = HOMOGLYPH_PATTERN.findall(row['full_text'])
    print(f"\nMatched misspellings: {misspellings}")
    print(f"Matched homoglyphs: {homoglyphs}")
    print()

Ham emails triggering Spelling/Grammar: 0/5



## 10. Debug: Inspect URLs from Phishing Emails Not Detected

In [33]:
# Get phishing emails that didn't trigger suspicious link cue
phishing_sample = sample_df[sample_df['expected_class'] == 1].copy()
phishing_sample['link_triggered'] = phishing_sample.apply(
    lambda row: check_suspicious_links(
        ast.literal_eval(row['extracted_urls']) if isinstance(row['extracted_urls'], str) else row['extracted_urls']
    ),
    axis=1
)

missed_phishing = phishing_sample[phishing_sample['link_triggered'] == 0]
print(f"Phishing emails NOT detected by Suspicious Link: {len(missed_phishing)}/{len(phishing_sample)}\n")

for idx, row in missed_phishing.iterrows():
    print(f"{'='*80}")
    print(f"Source: {row['source']}")
    print(f"Subject: {row['subject']}")
    
    # Parse URLs
    links = row['extracted_urls']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except Exception:
            links = []
    
    print(f"Number of URLs: {len(links)}")
    
    if links:
        for i, link in enumerate(links[:3], 1):  # Show first 3 URLs
            print(f"\n  URL {i}: {link}")
            try:
                parsed = urlparse(link)
                print(f"    Host: {parsed.netloc}")
                print(f"    Path: {parsed.path}")
                
                # Check each condition
                host_lower = parsed.netloc.lower()
                path_lower = parsed.path.lower()
                
                print(f"    Shortener: {bool(SHORTENER_PATTERN.search(link))}")
                print(f"    Suspicious TLD: {bool(SUSPICIOUS_TLD_PATTERN.search(link))}")
                
                brand_match = False
                for brand in BRAND_KEYWORDS:
                    if brand in host_lower and host_lower not in ALLOWED_BRAND_DOMAINS:
                        brand_match = True
                        print(f"    Brand mismatch: {brand} in host but not in allowed domains")
                        break
                if not brand_match:
                    print(f"    Brand mismatch: False")
                
                suspicious_path = SUSPICIOUS_PATH_PATTERN.search(path_lower) if host_lower not in ALLOWED_BRAND_DOMAINS else False
                print(f"    Suspicious path: {bool(suspicious_path)}")
                
                hyphen_check = any(x in host_lower for x in ['verify', 'secure', 'account', 'login']) and '-' in host_lower
                print(f"    Hyphenated security: {hyphen_check}")
                
            except Exception as e:
                print(f"    Error parsing: {e}")
    else:
        print("  No URLs found in this email")
    
    print()

Phishing emails NOT detected by Suspicious Link: 5/15

Source: phishbowl
Subject: ESSENTIAL TASK:
Number of URLs: 0
  No URLs found in this email

Source: phishbowl
Subject: Dear [name or netid], kindly re-authenticate your 2FA settings | Sun, October 8, 2023
Number of URLs: 0
  No URLs found in this email

Source: phishbowl
Subject: New Message from Randy [Surname]
Number of URLs: 1

  URL 1: https://netid.cornell.edu.
    Host: netid.cornell.edu.
    Path: 
    Shortener: False
    Suspicious TLD: False
    Brand mismatch: False
    Suspicious path: False
    Hyphenated security: False

Source: phishbowl
Subject: Review and sign
Number of URLs: 1

  URL 1: https://netid.cornell.edu.
    Host: netid.cornell.edu.
    Path: 
    Shortener: False
    Suspicious TLD: False
    Brand mismatch: False
    Suspicious path: False
    Hyphenated security: False

Source: phishbowl
Subject: INDIVIDUAL ASSESSMENT REPORT
Number of URLs: 1

  URL 1: https://netid.cornell.edu.
    Host: netid.cornell

## 11. CRITICAL: Inspect Phishbowl Emails (Real-World Phishing)

In [34]:
# Get all 5 Phishbowl samples and see what they actually contain
phishbowl_sample = sample_df[sample_df['source'] == 'phishbowl'].copy()

print(f"PHISHBOWL EMAILS (Real-World Phishing): {len(phishbowl_sample)}\n")

for idx, row in phishbowl_sample.iterrows():
    print(f"{'='*80}")
    print(f"SUBJECT: {row['subject']}")
    print(f"\nSENDER: {row['sender']}")
    print(f"\nBODY (first 500 chars):\n{row['body'][:500]}")
    
    # Parse URLs
    links = row['extracted_urls']
    if isinstance(links, str):
        try:
            links = ast.literal_eval(links)
        except Exception:
            links = []
    
    print(f"\n--- URLS: {len(links)} ---")
    for link in links[:3]:
        print(f"  {link}")
    
    print(f"\n--- CUE TRIGGERS ---")
    text = row['full_text']
    sender = str(row['sender'])
    
    print(f"Suspicious Sender: {check_suspicious_sender(sender)}")
    print(f"Generic Greeting: {bool(GENERIC_GREETING_PATTERN.search(text))}")
    print(f"Spelling/Grammar: {check_spelling_grammar(text)}")
    print(f"Urgency: {bool(URGENCY_PATTERN.search(text))}")
    print(f"Threats: {bool(THREATS_PATTERN.search(text))}")
    print(f"Emotional Appeal: {check_emotional_appeal(text)}")
    print(f"Too Good To Be True: {bool(TOO_GOOD_TRUE_PATTERN.search(text))}")
    print(f"Personal Info: {bool(PERSONAL_INFO_PATTERN.search(text))}")
    print(f"Suspicious Link: {check_suspicious_links(links)}")
    
    print()

PHISHBOWL EMAILS (Real-World Phishing): 5

SUBJECT: ESSENTIAL TASK:

SENDER: [Unknown]

BODY (first 500 chars):
Hi [Name], Is your schedule open today? I am heading for a meeting, but I am excellent with emails if that works well for you as well. I'd want you to discreetly run a task for me. Thanks. -- [Spoofed Name] [Org] HR Manager ###-###-2251 ext ### Cornell Cooperative Extension [County Name] County [Address] TEL: (###) ###-2251 FAX: (###) ###-5148 [user]@cornell.edu

--- URLS: 0 ---

--- CUE TRIGGERS ---
Suspicious Sender: 0
Generic Greeting: False
Spelling/Grammar: 0
Urgency: False
Threats: False
Emotional Appeal: 0
Too Good To Be True: False
Personal Info: False
Suspicious Link: 0

SUBJECT: Dear [name or netid], kindly re-authenticate your 2FA settings | Sun, October 8, 2023

SENDER: [Unknown]

BODY (first 500 chars):
Setting up Microsoft's Multi-Factor Authentication (2FA). Please review your Multi-Factor Authentication (2FA) settings. Follow the steps below to conduct a revie